# IPI-persistence — single-model test on Kaggle (Qwen2.5-3B-Instruct)

Runs the pipeline (Phase A → states → Phase B → clean → judge) on **one open-weight model**, locally on
the Kaggle GPU, then prints per-condition results. **No API keys, no cost.**

**Before running — set in the right sidebar:**
- **Accelerator:** GPU T4 x1
- **Internet:** On  (needed to clone the repo and download the model)

**Checkpointing / resuming (important — Kaggle sessions can time out):**
- Everything is written to `/kaggle/working/runs` and every model call is cached by an input hash,
  so the run is **resumable**: re-running skips work already done.
- To resume after a timeout: **Save Version** (commit) so `/kaggle/working` is saved as output → open a
  **new version** and attach this notebook's output as an *input dataset* → the restore cell below copies
  `runs/` back → run again; finished calls come from the cache.
- A `runs_checkpoint.zip` is written after every step for easy download.

This notebook is set to the **full 60+12-scenario benchmark** (`LIMIT = None` in the run cell). It takes
**~3–6 h on a 3B model**, so run it as a committed version and rely on the checkpoint/resume flow if the
session ends. For a quick wiring test, set `LIMIT = 3` instead.

Note: this is a *coverage* run over all scenarios with light settings (1 repetition). It is not yet the
full statistical design in the preregistration (which uses more repetitions, `clean_reps: 5`, and
`n_subsequent_turns: 6`) — bump those in Cell 4 when you move to the real experiment.

In [ ]:
# --- Cell 1: get the repo + restore any previous checkpoint ---
import os, sys, subprocess, glob, shutil

REPO_URL  = "https://github.com/kritimathurug23-cpu/ipi-persistence.git"
REPO_DIR  = "/kaggle/working/ipi-persistence"
WORK_RUNS = "/kaggle/working/runs"

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    if r.returncode != 0:
        # Fallback: repo added as a Kaggle dataset (use if the GitHub repo is private)
        cands = glob.glob("/kaggle/input/*/ipi-persistence") + glob.glob("/kaggle/input/*/*/ipi-persistence")
        assert cands, "Clone failed and no repo dataset found. Enable Internet, or add the repo as a dataset."
        shutil.copytree(cands[0], REPO_DIR)
        print("Copied repo from", cands[0])
print("Repo:", REPO_DIR, "->", sorted(os.listdir(REPO_DIR))[:10])

# Restore a checkpoint from a previously-saved output attached as an input dataset
if not os.path.isdir(WORK_RUNS):
    restored = False
    for cand in glob.glob("/kaggle/input/*/runs") + glob.glob("/kaggle/input/*/*/runs"):
        shutil.copytree(cand, WORK_RUNS); print("Restored checkpoint from", cand); restored = True; break
    if not restored:
        print("No prior checkpoint found - starting fresh.")
else:
    print("Existing checkpoint in", WORK_RUNS)

In [ ]:
# --- Cell 2: dependencies (torch is preinstalled on Kaggle GPU) ---
!pip -q install pyyaml pandas numpy scipy statsmodels matplotlib lifelines langdetect scikit-learn tabulate
!pip -q install -U "transformers>=4.44" accelerate
print("dependencies ready")

In [ ]:
# --- Cell 3: load the model and route provider 'local_hf' to it (no repo edits, no API keys) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # ungated; swap for any HF instruct model that fits the GPU
print("Loading", MODEL_ID, "...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

@torch.inference_mode()
def _local_hf(model_id, system, messages, settings):
    msgs = ([{"role": "system", "content": system}] if system else []) + list(messages)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    max_new = int(settings.get("max_tokens", settings.get("max_new_tokens", 512)))
    temp = settings.get("temperature", None)
    gen = dict(max_new_tokens=max_new, pad_token_id=tokenizer.pad_token_id)
    if temp and float(temp) > 0:
        gen.update(do_sample=True, temperature=float(temp), top_p=float(settings.get("top_p", 1.0)))
    else:
        gen.update(do_sample=False)                       # deterministic (greedy) by default
    out = model.generate(**enc, **gen)
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return {"text": text, "usage": {"input": int(enc["input_ids"].shape[1]), "output": int(gen_ids.shape[0])}}

sys.path.insert(0, REPO_DIR)
import harness.providers as P
_orig_chat = P.chat
def _chat(provider, model_id, system, messages, settings, base_url=None, api_key_env=None):
    if provider == "local_hf":
        return _local_hf(model_id, system, messages, settings)
    return _orig_chat(provider, model_id, system, messages, settings, base_url, api_key_env)
P.chat = _chat                                            # harness.calls calls providers.chat dynamically
print("Model ready; provider 'local_hf' patched in.")

In [ ]:
# --- Cell 4: write a small Kaggle config and smoke-test one generation ---
import yaml
CFG = {
    "run_dir": WORK_RUNS + "/kaggle",           # absolute -> lands in persistent /kaggle/working
    "scenarios_dir": "data/scenarios",
    "unrelated_tasks": "data/unrelated_tasks.json",
    "system_prompt": "You are a helpful assistant.",
    "n_subsequent_turns": 4,
    "reps_phase_a": 1, "reps_phase_b": 1, "clean_reps": 1,
    "clean_summary_control": False,             # keep the test light (set True to include CLEAN_P3)
    "target_states_per_model": 6, "constructed_max_attempts": 2,
    "cross_task_conditions": ["P2", "P3"], "default_task_type": "same_topic",
    "models": [{"name": "qwen3b", "provider": "local_hf", "model_id": MODEL_ID,
                "settings": {"max_tokens": 256}}],
    "judge_model": {"name": "judge", "provider": "local_hf", "model_id": MODEL_ID,
                    "settings": {"max_tokens": 128, "temperature": 0}},
    "judge_max_rows": 20,
}
CFG_PATH = os.path.join(REPO_DIR, "config", "kaggle.yaml")
with open(CFG_PATH, "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)
print("Wrote", CFG_PATH)

In [ ]:
# --- Cell 5: run the pipeline (resumable). Each step checkpoints; re-running skips finished work. ---
os.chdir(REPO_DIR)
LIMIT = None   # full 60+12-scenario benchmark. Set to a small int (e.g. 3) for a quick wiring test.

from harness.run import Ctx, phase_a, build_states, phase_b, clean, judge
ctx = Ctx(CFG_PATH, models_filter=None, limit=LIMIT)

for name, fn in [("phase-a", phase_a), ("states", build_states),
                 ("phase-b", phase_b), ("clean", clean), ("judge", judge)]:
    print(f"\n===== {name} =====", flush=True)
    fn(ctx)                                              # cached calls + existing rows are skipped
    shutil.make_archive("/kaggle/working/runs_checkpoint", "zip", WORK_RUNS)   # checkpoint after each step
print("\nDone. Checkpoint: /kaggle/working/runs_checkpoint.zip")

In [ ]:
# --- Cell 6: results (reads the saved results; no model needed) ---
import pandas as pd
from harness.calls import read_jsonl

RES = CFG["run_dir"] + "/results"
pb = pd.DataFrame(read_jsonl(RES + "/phase_b.jsonl"))
print("phase_b rows:", len(pb))
if len(pb):
    d = pb[pb.task_type == "same_topic"].copy()
    d["marker_f"] = d.marker.astype(float)                      # convert first, then group
    print("\nAttacker-marker rate by condition (mean over turns, %):")
    print((d.groupby("condition").marker_f.mean() * 100).round(1).to_string())
    ck = d.dropna(subset=["constraint_kept"]).copy()
    if len(ck):
        ck["ck"] = ck.constraint_kept.astype(float)
        print("\nConstraint-kept / RQ2 rate by condition (%):")
        print((ck.groupby("condition").ck.mean() * 100).round(1).to_string())

## Notes

- **Model:** `Qwen/Qwen2.5-3B-Instruct` (ungated, fits a T4). Change `MODEL_ID` in Cell 3 for any HF
  instruct model; larger ones may need 4-bit (`bitsandbytes`) or a bigger accelerator.
- **This is a wiring test.** With `LIMIT = 3` and 1 repetition the numbers are not meaningful — they
  only prove the pipeline runs end-to-end on a real model. Set `LIMIT = None` for the full benchmark.
- **Checkpoint / resume:** results and the per-call cache live in `/kaggle/working/runs`. If the session
  ends, **Save Version**, then in a new version attach this notebook's output as an input dataset — Cell 1
  restores it and Cell 5 continues from where it stopped (finished calls are served from the cache).
- **Full analysis (optional):** to build the tables/figures/report, run in a cell:
  `!cd {REPO_DIR} && python -m analysis.analyze --config config/kaggle.yaml`
  (some tables need more than a tiny test set).